# Análise de dados TCP-CII

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,TTASGKLVTQWCCR,301,314,14,HLA-DRB3*02:02,129,100.00,SGKLVTQWC,0.000011,100.00
16412,1,TTASGKLVTQWCCRSCTMPP,301,320,20,HLA-DRB3*02:02,535,100.00,LVTQWCCRS,0.000011,100.00
16413,1,KLVTQWCCRSCTMPPLRFLGE,306,326,21,HLA-DRB1*04:01,603,100.00,CRSCTMPPL,0.000010,100.00
16414,1,TTASGKLVTQWCC,301,313,13,HLA-DRB3*02:02,61,100.00,TASGKLVTQ,0.000006,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [3]:
df_mbp_m5 = df[df['median binding percentile'] < 1].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
86,1,GSGIFVVDNVHTWTEQYKF,16,34,19,HLA-DRB3*01:01,411,0.90,FVVDNVHTW,0.349148,0.90
87,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB3*02:02,314,0.93,WQIEKASLI,0.454709,0.93
88,1,QKAVHADMGYWIES,191,204,14,HLA-DRB3*01:01,107,0.95,VHADMGYWI,0.432979,0.95
89,1,AAIKDQKAVHADM,186,198,13,HLA-DRB1*01:01,38,0.98,IKDQKAVHA,0.608534,0.98


## Agrupando por pepitideos e agregando colunas pertinentes.

In [4]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADM,186,198,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01"
1,AAIKDQKAVHADMG,186,199,1,0.720,HLA-DRB4*01:01
2,AKIFTPEARNSTF,121,133,1,0.800,HLA-DRB1*11:01
3,ERRAWNSLEVEDY,146,158,1,0.490,HLA-DQA1*05:01/DQB1*02:01
4,ERRAWNSLEVEDYG,146,159,1,0.260,HLA-DQA1*05:01/DQB1*02:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGK,256,272,1,0.520,HLA-DQA1*05:01/DQB1*02:01
76,YRQGYATQTVGPWHLGKL,256,273,1,0.510,HLA-DQA1*05:01/DQB1*02:01
77,YRQGYATQTVGPWHLGKLE,256,274,1,0.660,HLA-DQA1*05:01/DQB1*02:01
78,YRQGYATQTVGPWHLGKLEI,256,275,1,0.820,HLA-DQA1*05:01/DQB1*02:01


In [5]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,KNQTWQIEKASLIEV,206,220,1,0.06,HLA-DRB1*01:01
1,KNQTWQIEKASLIEVKTC,206,223,1,0.06,HLA-DRB1*01:01
2,WIESSKNQTWQIEKASLIEVK,201,221,1,0.07,HLA-DRB1*01:01
3,KNQTWQIEKASLIEVKTCL,206,224,1,0.08,HLA-DRB1*01:01
4,KNQTWQIEKASLIEVKTCLW,206,225,1,0.11,HLA-DRB1*01:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGKLEID,256,276,1,0.86,HLA-DQA1*05:01/DQB1*02:01
76,HRLMSAAIKDQKAVHADMG,181,199,1,0.87,HLA-DRB4*01:01
77,GSGIFVVDNVHTWTEQYKF,16,34,1,0.90,HLA-DRB3*01:01
78,QKAVHADMGYWIES,191,204,1,0.95,HLA-DRB3*01:01


In [6]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,KNQTWQIEKASLIEV,206,220,1,0.06,HLA-DRB1*01:01
1,KNQTWQIEKASLIEVKTC,206,223,1,0.06,HLA-DRB1*01:01
2,WIESSKNQTWQIEKASLIEVK,201,221,1,0.07,HLA-DRB1*01:01
3,KNQTWQIEKASLIEVKTCL,206,224,1,0.08,HLA-DRB1*01:01
4,KNQTWQIEKASLIEVKTCLW,206,225,1,0.11,HLA-DRB1*01:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGKLEID,256,276,1,0.86,HLA-DQA1*05:01/DQB1*02:01
76,HRLMSAAIKDQKAVHADMG,181,199,1,0.87,HLA-DRB4*01:01
77,GSGIFVVDNVHTWTEQYKF,16,34,1,0.90,HLA-DRB3*01:01
78,QKAVHADMGYWIES,191,204,1,0.95,HLA-DRB3*01:01


## Filtragem por qte_de_alelos

In [7]:
filtarar_por_qte_de_alelos = 2

In [8]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= filtarar_por_qte_de_alelos
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YRQGYATQTVGPWH,256,269,2,0.385,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,QYKFQPESPARLA,31,43,2,0.455,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
2,YRQGYATQTVGPW,256,268,2,0.465,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
3,KNQTWQIEKASLIE,206,219,2,0.480,"HLA-DRB1*01:01, HLA-DRB1*07:01"
4,KNQTWQIEKASLIEVKT,206,222,2,0.485,"HLA-DRB1*01:01, HLA-DRB3*02:02"
5,QYKFQPESPARLASA,31,45,2,0.505,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
6,QYKFQPESPARLAS,31,44,2,0.520,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
7,KNQTWQIEKASLIEVK,206,221,3,0.700,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB3*02:02"
8,AAIKDQKAVHADM,186,198,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01"
9,HRLMSAAIKDQKA,181,193,2,0.755,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [9]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0       YRQGYATQTVGPWH
1        QYKFQPESPARLA
2        YRQGYATQTVGPW
3       KNQTWQIEKASLIE
4    KNQTWQIEKASLIEVKT
5      QYKFQPESPARLASA
6       QYKFQPESPARLAS
7     KNQTWQIEKASLIEVK
8        AAIKDQKAVHADM
9        HRLMSAAIKDQKA
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [10]:
!seqkit grep -s -v -r -p '[-*X?]' './Fastas/denv4_NS1_final.fasta' > DENV4_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [11]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,NaN
1,2,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,NaN
2,3,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN
3,4,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN
4,5,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN
...,...,...,...,...,...,...,...,...
114,115,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN
115,116,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN
116,117,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN
117,118,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN


### Merge da colunas qte_de_alelos e alelos ao dataframe conservacy_result

In [12]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,2.0,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2.0,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
2,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN,NaN
3,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN,NaN
4,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN,NaN
...,...,...,...,...,...,...,...,...
114,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN,NaN
115,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN,NaN
116,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN,NaN
117,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN,NaN


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [13]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,2.0,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",62.42
1,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2.0,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1...",95.30
2,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN,NaN,71.81
3,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN,NaN,62.42
4,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN,NaN,95.30
...,...,...,...,...,...,...,...,...,...
114,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN,NaN,29.53
115,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN,NaN,88.59
116,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN,NaN,83.89
117,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN,NaN,24.16


### Sort e filtragem por percent_match e presença em alelos

In [14]:
# Parametros
percent_match_minimo = 95.0
filtarar_por_qte_de_alelos = 5

In [15]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= percent_match_minimo]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 9,QYKFQPESPARLA,13,98.66% (147/149),84.62%,100.00%,2.0,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
1,NP 18,QYKFQPESPARLAS,14,98.66% (147/149),78.57%,100.00%,2.0,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
2,NP 96,PSLRTTTASGKLV,13,98.66% (147/149),92.31%,100.00%,NaN,NaN,98.66
3,NP 15,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,2.0,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
4,NP 48,QYKFQPESPARLASAI,16,97.99% (146/149),75.00%,100.00%,NaN,NaN,97.99
5,NP 95,ADMGYWIESSKNQT,14,97.99% (146/149),78.57%,100.00%,NaN,NaN,97.99
6,NP 97,HTWTEQYKFQPESPARLA,18,97.99% (146/149),88.89%,100.00%,NaN,NaN,97.99
7,NP 113,HTWTEQYKFQPESPARL,17,97.99% (146/149),88.24%,100.00%,NaN,NaN,97.99
8,NP 114,HTWTEQYKFQPESPAR,16,97.99% (146/149),93.75%,100.00%,NaN,NaN,97.99
9,NP 49,HTWTEQYKFQPESPARLASA,20,97.99% (146/149),80.00%,100.00%,NaN,NaN,97.99
